## 📊Data Science - CA3 - Regression

## 📦 Imports

In [15]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

## 📂LOAD DATA

In [ ]:
train_df = pd.read_excel("./task 2 - datasets/train.xlsx")
test_df = pd.read_excel("./task 2 - datasets/test.xlsx")

Train shape: (30130, 26)
Test shape: (7533, 25)


## 🛠️FEATURE ENGINEERING

In [ ]:
def create_features(df):
    df = df.copy()

    df["total_media"] = df["image_count"] + df["video_count"]
    df["internal_link_ratio"] = df["internal_link_count"] / (df["link_count"] + 1)

    df["sentiment_balance"] = df["positive_word_rate"] - df["negative_word_rate"]

    df["content_strength"] = np.abs(df["content_polarity"])
    df["title_strength"] = np.abs(df["title_polarity"])

    df["content_richness"] = df["word_count"] * df["unique_word_rate"]

    df["media_density"] = df["total_media"] / (df["word_count"] + 1)
    df["keyword_density"] = df["keyword_count"] / (df["word_count"] + 1)

    df["log_word_count"] = np.log1p(df["word_count"])
    df["log_keyword_count"] = np.log1p(df["keyword_count"])

    df["engagement_score"] = df["word_count"] * df["keyword_density"]

    return df

train_df = create_features(train_df)
test_df = create_features(test_df)


### 📌
This section creates new useful features from the original columns.
For example, total_media combines images and videos, media_density measures how media-rich an article is compared to its length, and sentiment_balance measures the difference between positive and negative words. These engineered features help the model understand article quality, richness, and emotional strength better. 


## 🎯 Target Preparation


In [ ]:
y = np.log1p(train_df["shares"])

X = train_df.drop(columns=["shares", "id"])
X_test = test_df.drop(columns=["id"])


### 📌
In this section, the target column shares is transformed using log1p.
This is important because the evaluation metric is RMSLE, which measures error on a logarithmic scale. The id column is removed because it is only an identifier, and shares is removed from the input features because it is the value we want to predict.

## 🔢ONE-HOT

In [ ]:
cat_cols = X.select_dtypes(include=["object"]).columns

X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

X = X.fillna(X.median())
X_test = X_test.fillna(X_test.median())

### 📌
This section converts categorical columns into numeric columns using one-hot encoding.
Machine learning models need numerical input, so text categories such as article channels must be converted into numbers. The train and test datasets are then aligned to make sure they have exactly the same feature columns. 

## 📏 RMSLE Metric

In [16]:
def rmsle(y_true, y_pred):
    y_true = np.expm1(y_true)
    y_pred = np.expm1(y_pred)
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))


### 📌
This section defines the RMSLE evaluation function.
RMSLE is suitable for this task because it focuses on relative prediction errors instead of only absolute errors. It is especially useful when the target variable, such as article shares, is skewed. 

## 🤖MODEL FACTORY (MULTI-SEED)

In [17]:
def get_models(seed):
    return [
        XGBRegressor(
            n_estimators=1200,
            learning_rate=0.02,
            max_depth=6,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_weight=3,
            gamma=0.1,
            tree_method="hist",
            random_state=seed
        ),
        LGBMRegressor(
            n_estimators=1600,
            learning_rate=0.02,
            num_leaves=90,
            subsample=0.85,
            colsample_bytree=0.85,
            min_child_samples=20,
            random_state=seed
        ),
        CatBoostRegressor(
            iterations=1600,
            depth=6,
            learning_rate=0.02,
            loss_function="RMSE",
            verbose=0,
            random_state=seed
        )
    ]


### 📌
This section defines the main regression models used in the project.
The models include XGBoost, LightGBM, and CatBoost, which are powerful traditional machine learning models for tabular data. Each model is created with the same random seed to make the results more stable and reproducible.

## 🔁OOF + PSEUDO LABEL

In [18]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def train_oof(models):

    oof_train = np.zeros((X.shape[0], len(models)))
    oof_test = np.zeros((X_test.shape[0], len(models)))

    for i, model in enumerate(models):

        test_preds = np.zeros((X_test.shape[0], 5))

        for fold, (tr, val) in enumerate(kf.split(X)):

            X_tr, X_val = X.iloc[tr], X.iloc[val]
            y_tr, y_val = y.iloc[tr], y.iloc[val]

            model.fit(X_tr, y_tr)

            oof_train[val, i] = model.predict(X_val)
            test_preds[:, fold] = model.predict(X_test)

        oof_test[:, i] = test_preds.mean(axis=1)

    return oof_train, oof_test


### 📌

This section uses 5-Fold Cross Validation.
The training data is split into five parts. In each round, four parts are used for training and one part is used for validation. This gives a more reliable evaluation than using only one train-validation split. 

## 🌱MULTI-SEED ENSEMBLE

In [19]:
all_oof_train = []
all_oof_test = []

print("\nTraining multi-seed ensemble...")

for seed in [42, 202, 999]:

    models = get_models(seed)
    oof_tr, oof_te = train_oof(models)

    all_oof_train.append(oof_tr)
    all_oof_test.append(oof_te)



Training multi-seed ensemble...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007854 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6345
[LightGBM] [Info] Number of data points in the train set: 24104, number of used features: 40
[LightGBM] [Info] Start training from score 7.345058
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003476 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6348
[LightGBM] [Info] Number of data points in the train set: 24104, number of used features: 40
[LightGBM] [Info] Start training from score 7.341264
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 6353
[LightGBM] [Info] Number of data points in the train set: 24104, number of used feature

### 📌
This section trains the models using multiple random seeds.
Using different seeds makes the predictions more stable because tree-based models can slightly change their results depending on randomness. Averaging predictions from several seeds helps reduce variance and improves generalization.

## 🧱STACK FEATURES

In [20]:
oof_train_final = np.mean(all_oof_train, axis=0)
oof_test_final = np.mean(all_oof_test, axis=0)


## 🧮META MODEL

In [21]:
meta = Ridge(alpha=1.0)
meta.fit(oof_train_final, y)

final_pred_log = meta.predict(oof_test_final)

## 🏁FINAL OUTPUT

In [22]:
final_pred = np.expm1(final_pred_log)
final_pred = np.maximum(final_pred, 0)

submission = pd.DataFrame({
    "id": test_df["id"],
    "shares": final_pred.astype(int)
})

submission.to_csv("submission.csv", index=False)

print("\n FINAL BOSS DONE - submission saved.")


 FINAL BOSS DONE - submission saved.
